In [1]:
import os
import pandas as pd
import rasterio

import rsutils.modify_images
import rsutils.utils

from fetch_satdata.workflows.demo_model_deploy import demo_model_deploy

In [2]:
INFERENCE_DATACUBE_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/inference_datacubes'
OUTPUT_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/model_outputs'
MERGED_OUTPUT_FILEPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/model_outputs/merged.tif'

NJOBS = 8
NODATA = 255

In [3]:
os.makedirs(OUTPUT_FOLDERPATH, exist_ok=True)

INFERENCE_DATACUBE_CATALOG_FILEPATH = os.path.join(INFERENCE_DATACUBE_FOLDERPATH, 'input.csv')
INPUT_CSV_FILEPATH = os.path.join(OUTPUT_FOLDERPATH, 'input.csv')

In [4]:
input_df = pd.read_csv(INFERENCE_DATACUBE_CATALOG_FILEPATH)

input_df = input_df.rename(columns={'export_folderpath': 'datacube_folderpath'})

input_df['output_filepath'] = input_df['datacube_folderpath'].apply(
    lambda x: os.path.join(OUTPUT_FOLDERPATH, x.split('/')[-1], 'output.tif')
)

input_df = input_df[['datacube_folderpath', 'output_filepath']]

input_df.to_csv(INPUT_CSV_FILEPATH, index=False)

In [ ]:
demo_model_deploy(
    input_csv_filepath = INPUT_CSV_FILEPATH, 
    cores = NJOBS,
    # dry_run = True
)

In [6]:
def merge_images(
    filepaths:list[str],
    output_filepath:str,
    njobs = NJOBS,
    nodata = NODATA,
):
    data_profile_list = rsutils.modify_images.load_images(
        src_filepaths = filepaths,
        njobs = njobs,
    )

    merged_data, merged_profile = rsutils.modify_images.merge_inplace(
        data_profile_list = data_profile_list,
        nodata = nodata,
    )

    merged_profile = rsutils.utils.driver_specific_meta_updates(
        meta=merged_profile, driver='GTiff'
    )

    with rasterio.open(output_filepath, 'w', **merged_profile) as dst:
        dst.write(merged_data)

In [ ]:
merge_images(filepaths = input_df['output_filepath'], output_filepath = MERGED_OUTPUT_FILEPATH)

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
import rasterio


def plot_categorical_tif(
    tif_path,
    value_to_label,
    value_to_color,
    nodata_value,
    title=None,
    save_path=None,
    dpi=300,
    scale=8,
):
    """
    value_to_label: dict {int: str}
    value_to_color: dict {int: "#RRGGBB"}
    nodata_value: int or float
    """

    # --- read raster ---
    with rasterio.open(tif_path) as src:
        arr = src.read(1)

    # --- mask nodata ---
    arr = np.ma.masked_equal(arr, nodata_value)

    # --- ensure consistent ordering ---
    values = sorted(value_to_label.keys())

    # --- build colormap ---
    colors = [value_to_color[v] for v in values]
    cmap = ListedColormap(colors)

    # boundaries ensure discrete coloring
    bounds = np.array(values + [values[-1] + 1]) - 0.5
    norm = BoundaryNorm(bounds, cmap.N)

    # --- plot ---
    fig, ax = plt.subplots(figsize=(scale, scale))

    im = ax.imshow(arr, cmap=cmap, norm=norm)

    # remove axes for clean map-style output
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=14)

    # --- legend (top-left) ---
    legend_handles = [
        mpatches.Patch(color=value_to_color[v], label=value_to_label[v])
        for v in values
    ]

    # --- legend outside (top-right) ---
    ax.legend(
        handles=legend_handles,
        loc="upper left",
        bbox_to_anchor=(1.02, 1),  # push outside to the right
        borderaxespad=0,
        frameon=True,
        fontsize=10,
        title="Legend",
        title_fontsize=11,
    )

    # --- make space for legend ---
    plt.subplots_adjust(right=0.75)

    plt.tight_layout()

    # --- save high quality ---
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
        # also save vector version for papers
        plt.savefig(save_path.replace(".png", ".pdf"), bbox_inches="tight")

    plt.show()

In [ ]:
plot_categorical_tif(
    tif_path = MERGED_OUTPUT_FILEPATH,
    value_to_label = {
        0: 'alfalfa_lucerne',
        1: 'buckwheat',
        2: 'grain_maize_corn_popcorn',
        3: 'hemp_cannabis',
        4: 'mustard',
        5: 'pasture_meadow_grassland_grass',
        6: 'spring_common_soft_wheat',
        7: 'spring_durum_hard_wheat',
        8: 'sunflower',
        9: 'winter_common_soft_wheat',
        10: 'winter_durum_hard_wheat',
    },
    value_to_color = {
        0: '#dfff7f',
        1: '#885f00',
        2: '#ffec39',
        3: '#23ff01',
        4: '#ff8e51',
        5: '#008020',
        6: '#b1585b',
        7: '#993f41',
        8: '#ff73fb',
        9: '#8bdbff',
        10: '#268fbc',
    },
    nodata_value = 255,
    scale = 10,
)

## Building a STAC catalog

In [ ]:
from pathlib import Path
import csv

import pandas as pd
import pystac
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import box, mapping

CSV_PATH = "rasters.csv"          # has column: filepath
OUT_DIR = Path("stac_catalog")    # output folder


def make_item(fp: str) -> pystac.Item:
    path = Path(fp).expanduser().resolve()

    with rasterio.open(path) as src:
        bounds_4326 = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
        geom = mapping(box(*bounds_4326))

        item = pystac.Item(
            id=path.stem,
            geometry=geom,
            bbox=list(bounds_4326),
            datetime=None,
            properties={},
        )

        asset = pystac.Asset(
            href=path.as_uri(),  # local file:///... URI
            media_type=pystac.MediaType.COG,
            roles=["data"],
            title=path.name,
        )
        item.add_asset("cog", asset)

        item.properties.update({
            "proj:epsg": src.crs.to_epsg() if src.crs else None,
            "proj:shape": [src.height, src.width],
            "proj:transform": list(src.transform)[:6],
        })

    return item


def build_catalog(raster_filepaths_df, filepath_col):
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    if "filepath" not in df.columns:
        raise ValueError('CSV must contain a "filepath" column')

    catalog = pystac.Catalog(id="local-cogs", description="Local STAC catalog for COG outputs")

    collection = pystac.Collection(
        id="local-cogs-collection",
        description="COGs generated locally",
        extent=pystac.Extent(
            spatial=pystac.SpatialExtent([[-180, -90, 180, 90]]),
            temporal=pystac.TemporalExtent([[None, None]]),
        ),
    )

    for fp in df["filepath"].dropna():
        item = make_item(fp)
        collection.add_item(item)

    catalog.add_child(collection)

    catalog.normalize_hrefs(str(OUT_DIR))
    catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)